In [ ]:
# TCP Echo server program
import socket, threading

clients=[] # 클라이언트 소켓 객체를 저장할 리스트
nicknames = [] # 클라이언트 닉네임을 저장할 리스트

# 메시지 수신되면 대화방에 참여한 전 클라이언트에게 전송
def broadcast(message, cl):
    for client in clients:
        if client == cl:
            continue
        client.sendall(message.encode('utf-8'))

# 클라이언트 수신된 메시지 처리
def recv(client):
    while True:
        try: # 정상적으로 메시지 수신되면
             # 수신된 메시지 broadcast로 전달
            rxmsg = client.recv(1024).decode()
            # 특정 클라이언트로 메시지 수신되면 전 클라이언트로 broadcast
            broadcast(rxmsg, client)
        except: # 네트워크 오류가 발생한 경우
            index = clients.index(client)
            clients.remove(client) # 클라이언트 정보삭제
            client.close()
            nickname = nicknames[index]
            broadcast(f"{nickname}님이 퇴장했습니다",_)
            nicknames.remove(nickname) # 닉네임 삭제
            print(f"{nickname}퇴장, {len(clients)}명 참가중")
            break

# 대화방에 참여하는 클라이언트 관리
def client_handle():
    while True:
        conn, addr = server.accept()
        print(f">> {addr} is connected")
        # 접속된 클라이언트에서 닉네임 요청 및 저장
        conn.sendall('NICK'.encode('utf-8'))
        nickname = conn.recv(1024).decode('utf-8')
        print(f"{nickname} 입장") # 서버용 출력
        nicknames.append(nickname) # 새로 접속한 클라이언트 닉네임 리스트에 저장
        clients.append(conn) # 새로 접속한 클라이언트 소켓객체를 리스트에 저장
        # 새로 접속된 클라이언트 정보를 모든 클라이언트에게 알람
        broadcast(f"{nickname}님이 입장했습니다",_)
        broadcast(f"{len(clients)}명이 대화방에 참가중입니다",_)
        conn.sendall(f"{nickname}님 반갑습니다".encode('utf-8'))
        # 새로 참가한 클라이언트의 수신 스레드 생성 및 실행
        receive = threading.Thread(target=recv, args=(conn,))
        receive.start()

if __name__=="__main__":
    IP = '0.0.0.0' # 내 컴퓨터에 배정된 모든 ip 수
    PORT = 9999
    address = (IP, PORT) # tuple로 주소정보 저장
    print(">> Group Chatting Server start")
    
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR,1)
    
    server.bind(address)
    
    server.listen() # 클라이언트 하나만 받음
    client_handle()

>> Group Chatting Server start
>> ('10.2.50.7', 63565) is connected
백인원1 입장
>> ('10.2.50.7', 63566) is connected
백인원2 입장
백인원2퇴장, 1명 참가중
백인원1퇴장, 0명 참가중
